# Stage 4 — YOLO11n-P2 + Small-Priority Sliced Fine-Tuning

This notebook is the next coral detector experiment after Stage 3.

## Goal

Improve the measured weak group:

- **Stage 2 Small Recall:** 0.3611
- **Stage 3 Small Recall:** 0.4167
- **Stage 3 Crowded Recall:** 0.6822
- **Stage 3 mAP50:95:** ~0.473

## Intervention

**Stage-3 best.pt → custom YOLO11n-P2 (strides 4/8/16/32) → transfer compatible weights → Small-Priority train-only slices → 960px fine-tune → original Validation audit → Stage2/Stage3/Stage4 comparison**

The new P2 branch is intended to preserve higher-resolution spatial features for very small coral instances.

## Experimental rules

- Train split only is used to generate slices.
- Validation remains the original **100 images / 354 GT boxes**.
- Internal test is **not loaded or evaluated**.
- Small definition remains **bbox area <1% of original image area** for direct comparability with the completed audit.
- Crowded definition remains **>=4 GT objects/image** for direct comparability.
- Current failed SAHI640 inference is **not part of this primary experiment**.


In [ ]:
# ============================================================
# CELL 1 — INSTALL + MOUNT + ENVIRONMENT
# ============================================================

!pip -q install ultralytics==8.4.117 albumentations==2.0.8 pycocotools pyyaml

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import defaultdict
import os
import json
import random
import shutil
import time
import math
import yaml

import numpy as np
import pandas as pd
import cv2
import torch
import albumentations as A
import ultralytics

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 100)
print("ENVIRONMENT")
print("=" * 100)
print("Ultralytics    :", ultralytics.__version__)
print("Albumentations :", A.__version__)
print("PyTorch        :", torch.__version__)
print("CUDA           :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU runtime before continuing.")

print("GPU            :", torch.cuda.get_device_name(0))

gpu_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU memory     : {gpu_gb:.2f} GB")

print("\nSTATUS: ENVIRONMENT_READY")


In [ ]:
# ============================================================
# CELL 2 — PROJECT PATHS + STRICT SOURCE CHECK
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Coral_Reef_AI_Project"
)

DETECTION_V1 = (
    PROJECT_ROOT
    / "01_Data"
    / "Final"
    / "Detection_v1"
)

TRAIN_IMG_DIR = DETECTION_V1 / "images" / "train"
VAL_IMG_DIR = DETECTION_V1 / "images" / "validation"

TRAIN_COCO = DETECTION_V1 / "annotations" / "instances_train.json"
VAL_COCO = DETECTION_V1 / "annotations" / "instances_validation.json"

STAGE3_BEST = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_Stage3_TargetedSliced_SmallCrowded"
    / "Detection_Stage3"
    / "weights"
    / "best.pt"
)

EXP_ROOT = (
    PROJECT_ROOT
    / "06_Experiments"
    / "YOLO11n_P2_Stage4_SmallPriority"
)

EXP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# Local working dataset for fast Colab I/O.
BUILD_ROOT = Path(
    "/content/Coral_YOLO11_P2_Stage4_SmallPriority"
)

for p in [
    TRAIN_IMG_DIR,
    VAL_IMG_DIR,
    TRAIN_COCO,
    VAL_COCO,
    STAGE3_BEST,
]:
    if not p.exists():
        raise FileNotFoundError(p)

with open(TRAIN_COCO, "r", encoding="utf-8") as f:
    train_coco = json.load(f)

with open(VAL_COCO, "r", encoding="utf-8") as f:
    val_coco = json.load(f)

print("=" * 100)
print("SOURCE DATA")
print("=" * 100)
print("Train images :", len(train_coco["images"]))
print("Train boxes  :", len(train_coco["annotations"]))
print("Val images   :", len(val_coco["images"]))
print("Val boxes    :", len(val_coco["annotations"]))
print("Stage3 best  :", STAGE3_BEST)

assert len(train_coco["images"]) == 452
assert len(train_coco["annotations"]) == 1540
assert len(val_coco["images"]) == 100
assert len(val_coco["annotations"]) == 354

print("\nTEST SPLIT WAS NOT LOADED.")
print("STATUS: SOURCE_CHECK_PASSED")


In [ ]:
# ============================================================
# CELL 3 — OLD CUSTOM CLASS NAMES FOR CHECKPOINT COMPATIBILITY
#
# Stage 3 normally no longer needs these transforms, but keeping
# the names available makes loading older chained checkpoints robust.
# They are NOT used as Stage-4 training augmentations.
# ============================================================

class UnderwaterTurbidity(A.ImageOnlyTransform):

    def __init__(
        self,
        alpha_range=(0.05, 0.16),
        blur_range=(3, 7),
        p=0.15,
    ):
        super().__init__(p=p)
        self.alpha_range = alpha_range
        self.blur_range = blur_range

    def apply(self, img, **params):
        alpha = np.random.uniform(*self.alpha_range)
        veil_value = np.random.randint(125, 180)
        veil = np.full_like(img, veil_value)
        result = cv2.addWeighted(
            img, 1.0 - alpha, veil, alpha, 0
        )
        kernel = int(np.random.choice([3, 5, 7]))
        return cv2.GaussianBlur(
            result, (kernel, kernel), 0
        )

    def get_transform_init_args_names(self):
        return ("alpha_range", "blur_range")


class UnderwaterBackscatter(A.ImageOnlyTransform):

    def __init__(
        self,
        particles_range=(20, 65),
        radius_range=(1, 3),
        opacity_range=(0.08, 0.20),
        p=0.15,
    ):
        super().__init__(p=p)
        self.particles_range = particles_range
        self.radius_range = radius_range
        self.opacity_range = opacity_range

    def apply(self, img, **params):
        h, w = img.shape[:2]
        overlay = np.zeros_like(img)

        n_particles = np.random.randint(
            self.particles_range[0],
            self.particles_range[1] + 1
        )

        for _ in range(n_particles):
            x = np.random.randint(0, w)
            y = np.random.randint(0, h)

            radius = np.random.randint(
                self.radius_range[0],
                self.radius_range[1] + 1
            )

            intensity = np.random.randint(160, 256)

            cv2.circle(
                overlay,
                (x, y),
                radius,
                (intensity, intensity, intensity),
                -1
            )

        overlay = cv2.GaussianBlur(
            overlay, (3, 3), 0
        )

        opacity = np.random.uniform(
            *self.opacity_range
        )

        return cv2.addWeighted(
            img, 1.0, overlay, opacity, 0
        )

    def get_transform_init_args_names(self):
        return (
            "particles_range",
            "radius_range",
            "opacity_range",
        )


print("Checkpoint compatibility classes: READY")


In [ ]:
# ============================================================
# CELL 4 — CREATE CUSTOM YOLO11n-P2 ARCHITECTURE YAML
#
# Standard YOLO11 Detect:
# P3/8, P4/16, P5/32
#
# Stage 4 Detect:
# P2/4, P3/8, P4/16, P5/32
# ============================================================

P2_YAML = EXP_ROOT / "yolo11n_p2_coral.yaml"

p2_yaml_text = r'''
# YOLO11n-P2 custom coral detector
# P2/4 - P5/32 outputs

nc: 1

# Force nano compound scale.
scale: n

scales:
  n: [0.50, 0.25, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]                # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]               # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]        # 2-P2 feature
  - [-1, 1, Conv, [256, 3, 2]]               # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]        # 4-P3 feature
  - [-1, 1, Conv, [512, 3, 2]]               # 5-P4/16
  - [-1, 2, C3k2, [512, True]]               # 6-P4 feature
  - [-1, 1, Conv, [1024, 3, 2]]              # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]              # 8
  - [-1, 1, SPPF, [1024, 5]]                 # 9
  - [-1, 2, C2PSA, [1024]]                   # 10

head:
  # P5 -> P4
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 11
  - [[-1, 6], 1, Concat, [1]]                    # 12
  - [-1, 2, C3k2, [512, False]]                  # 13

  # P4 -> P3
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 14
  - [[-1, 4], 1, Concat, [1]]                    # 15
  - [-1, 2, C3k2, [256, False]]                  # 16-P3/8

  # NEW: P3 -> P2
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 17
  - [[-1, 2], 1, Concat, [1]]                    # 18 cat backbone P2
  - [-1, 2, C3k2, [128, False]]                  # 19-P2/4 xsmall

  # PAN: P2 -> P3
  - [-1, 1, Conv, [128, 3, 2]]                   # 20
  - [[-1, 16], 1, Concat, [1]]                   # 21
  - [-1, 2, C3k2, [256, False]]                  # 22-P3/8 small

  # PAN: P3 -> P4
  - [-1, 1, Conv, [256, 3, 2]]                   # 23
  - [[-1, 13], 1, Concat, [1]]                   # 24
  - [-1, 2, C3k2, [512, False]]                  # 25-P4/16 medium

  # PAN: P4 -> P5
  - [-1, 1, Conv, [512, 3, 2]]                   # 26
  - [[-1, 10], 1, Concat, [1]]                   # 27
  - [-1, 2, C3k2, [1024, True]]                  # 28-P5/32 large

  # Detect P2, P3, P4, P5
  - [[19, 22, 25, 28], 1, Detect, [nc]]           # 29
'''

P2_YAML.write_text(
    p2_yaml_text.strip() + "\n",
    encoding="utf-8"
)

print(P2_YAML)
print("\n" + P2_YAML.read_text(encoding="utf-8"))
print("STATUS: P2_YAML_CREATED")


In [ ]:
# ============================================================
# CELL 5 — BUILD P2 MODEL + TRANSFER STAGE-3 WEIGHTS
# ARCHITECTURE SANITY CHECK BEFORE ANY TRAINING
# ============================================================

from ultralytics import YOLO

print("=" * 110)
print("BUILDING YOLO11n-P2")
print("=" * 110)

p2_model = YOLO(
    str(P2_YAML),
    task="detect"
)

print("\nArchitecture before weight transfer:")
p2_model.info()

# Official Ultralytics load() behavior loads only matching layers.
print("\n" + "=" * 110)
print("TRANSFERRING COMPATIBLE STAGE-3 WEIGHTS")
print("=" * 110)

p2_model.load(
    str(STAGE3_BEST)
)

print("\nArchitecture after transfer:")
p2_model.info()

# ------------------------------------------------------------
# STRICT STRIDE CHECK
# ------------------------------------------------------------

strides = [
    int(round(float(x)))
    for x in p2_model.model.stride.detach().cpu().tolist()
]

print("\nDetected output strides:", strides)

assert strides == [4, 8, 16, 32], (
    f"P2 architecture check failed. Expected [4,8,16,32], got {strides}"
)

# ------------------------------------------------------------
# PARAMETER COUNT
# ------------------------------------------------------------

P2_PARAMS = sum(
    p.numel()
    for p in p2_model.model.parameters()
)

print("P2 parameters:", f"{P2_PARAMS:,}")

# ------------------------------------------------------------
# LIGHT FORWARD SANITY CHECK
# Use 640 here only to verify graph/channels before training.
# Training itself remains 960.
# ------------------------------------------------------------

device_check = torch.device("cuda:0")

p2_model.model.to(device_check)
p2_model.model.eval()

with torch.no_grad():
    dummy = torch.zeros(
        1, 3, 640, 640,
        device=device_check
    )
    _ = p2_model.model(dummy)

print("\nForward pass: PASS")
print("P2 stride 4  : PASS")
print("P3 stride 8  : PASS")
print("P4 stride 16 : PASS")
print("P5 stride 32 : PASS")

print("\nSTATUS: P2_ARCHITECTURE_VALIDATED")


In [ ]:
# ============================================================
# CELL 6 — ANALYZE ORIGINAL TRAIN OBJECTS
# IDENTIFY TRUE SMALL OBJECTS BEFORE SLICING
# ============================================================

SMALL_AREA_RATIO = 0.01
MEDIUM_AREA_RATIO = 0.10
CROWDED_GT_COUNT = 4

train_images_by_id = {
    int(im["id"]): im
    for im in train_coco["images"]
}

anns_by_image = defaultdict(list)

for ann in train_coco["annotations"]:
    anns_by_image[int(ann["image_id"])].append(ann)

object_rows = []
image_rows = []

for image_id, im in train_images_by_id.items():

    W = float(im["width"])
    H = float(im["height"])
    image_area = max(W * H, 1.0)

    anns = anns_by_image[image_id]

    small_count = 0

    for ann in anns:

        x, y, bw, bh = map(float, ann["bbox"])
        rel_area = (bw * bh) / image_area

        if rel_area < SMALL_AREA_RATIO:
            size_bucket = "small"
            small_count += 1

        elif rel_area < MEDIUM_AREA_RATIO:
            size_bucket = "medium"

        else:
            size_bucket = "large"

        object_rows.append({
            "image_id": image_id,
            "file_name": Path(im["file_name"]).name,
            "annotation_id": int(ann["id"]),
            "bbox_x": x,
            "bbox_y": y,
            "bbox_w": bw,
            "bbox_h": bh,
            "relative_area": rel_area,
            "size_bucket": size_bucket,
        })

    image_rows.append({
        "image_id": image_id,
        "file_name": Path(im["file_name"]).name,
        "width": int(im["width"]),
        "height": int(im["height"]),
        "gt_count": len(anns),
        "small_gt_count": small_count,
        "has_small": small_count > 0,
        "crowded": len(anns) >= CROWDED_GT_COUNT,
    })

train_object_df = pd.DataFrame(object_rows)
train_image_df = pd.DataFrame(image_rows)

TRAIN_OBJECT_AUDIT = EXP_ROOT / "stage4_train_object_size_audit.csv"
TRAIN_IMAGE_AUDIT = EXP_ROOT / "stage4_train_image_priority_audit.csv"

train_object_df.to_csv(TRAIN_OBJECT_AUDIT, index=False)
train_image_df.to_csv(TRAIN_IMAGE_AUDIT, index=False)

print("=" * 100)
print("ORIGINAL TRAIN SIZE AUDIT")
print("=" * 100)

print(
    train_object_df["size_bucket"]
    .value_counts()
    .reindex(["small", "medium", "large"])
    .fillna(0)
    .astype(int)
)

print("\nImages with >=1 small object :", int(train_image_df["has_small"].sum()))
print("Crowded source images        :", int(train_image_df["crowded"].sum()))
print("Small definition             : <1% original image area")
print("Crowded definition           : >=4 GT objects/image")

assert len(train_object_df) == 1540
assert len(train_image_df) == 452

print("\nSTATUS: TRAIN_PRIORITY_ANALYSIS_COMPLETE")


In [ ]:
# ============================================================
# CELL 7 — GENERATE SMALL-PRIORITY SLICE CANDIDATES
#
# Priority design:
# 1) object-centered crops around ORIGINAL small GT
# 2) crowded-source grid crops
# 3) small number of other positives
# 4) hard/background negatives
#
# Slices are generated from TRAIN ONLY.
# ============================================================

SLICE_SIZE = 640
GRID_OVERLAP = 0.20
MIN_VISIBLE_FRACTION = 0.40

# Controlled caps so slices do not dominate original 452 images.
MAX_SMALL_SLICES = 350
MAX_CROWDED_SLICES = 150
MAX_OTHER_POSITIVE_SLICES = 50
MAX_NEGATIVE_SLICES = 60

# Center + mild jitter around each small object.
CENTER_JITTERS = [
    (0.00, 0.00),
    (0.12, 0.00),
    (-0.12, 0.00),
]

rng = random.Random(SEED)


def clamp(v, lo, hi):
    return max(lo, min(v, hi))


def make_centered_origin(cx, cy, W, H, tile_size, jx=0.0, jy=0.0):

    tw = min(tile_size, W)
    th = min(tile_size, H)

    target_cx = cx + jx * tw
    target_cy = cy + jy * th

    x0 = int(round(target_cx - tw / 2))
    y0 = int(round(target_cy - th / 2))

    x0 = int(clamp(x0, 0, max(W - tw, 0)))
    y0 = int(clamp(y0, 0, max(H - th, 0)))

    return x0, y0, int(tw), int(th)


def grid_starts(length, tile, overlap):

    if length <= tile:
        return [0]

    step = max(1, int(round(tile * (1.0 - overlap))))

    starts = list(range(0, max(length - tile + 1, 1), step))

    last = length - tile

    if starts[-1] != last:
        starts.append(last)

    return sorted(set(starts))


def clip_ann_to_tile(ann, tile_x, tile_y, tile_w, tile_h):

    x, y, bw, bh = map(float, ann["bbox"])

    x1 = x
    y1 = y
    x2 = x + bw
    y2 = y + bh

    tx1 = float(tile_x)
    ty1 = float(tile_y)
    tx2 = float(tile_x + tile_w)
    ty2 = float(tile_y + tile_h)

    cx1 = max(x1, tx1)
    cy1 = max(y1, ty1)
    cx2 = min(x2, tx2)
    cy2 = min(y2, ty2)

    clipped_w = max(0.0, cx2 - cx1)
    clipped_h = max(0.0, cy2 - cy1)

    original_area = max(bw * bh, 1e-9)
    visible_area = clipped_w * clipped_h
    visible_fraction = visible_area / original_area

    if (
        clipped_w <= 1.0
        or clipped_h <= 1.0
        or visible_fraction < MIN_VISIBLE_FRACTION
    ):
        return None

    return {
        "annotation_id": int(ann["id"]),
        "x": cx1 - tx1,
        "y": cy1 - ty1,
        "w": clipped_w,
        "h": clipped_h,
        "visible_fraction": visible_fraction,
    }


def candidate_from_origin(image_id, x0, y0, tw, th, reason):

    im = train_images_by_id[image_id]

    W = float(im["width"])
    H = float(im["height"])
    image_area = max(W * H, 1.0)

    kept = []

    for ann in anns_by_image[image_id]:

        clipped = clip_ann_to_tile(
            ann,
            x0,
            y0,
            tw,
            th
        )

        if clipped is None:
            continue

        _, _, bw, bh = map(float, ann["bbox"])
        original_rel_area = (bw * bh) / image_area

        clipped["original_relative_area"] = original_rel_area
        clipped["original_small"] = bool(
            original_rel_area < SMALL_AREA_RATIO
        )

        kept.append(clipped)

    return {
        "image_id": int(image_id),
        "file_name": Path(im["file_name"]).name,
        "x0": int(x0),
        "y0": int(y0),
        "tile_w": int(tw),
        "tile_h": int(th),
        "reason": reason,
        "annotations": kept,
        "contains_small": any(x["original_small"] for x in kept),
        "positive": len(kept) > 0,
        "kept_box_count": len(kept),
        "crowded_source": len(anns_by_image[image_id]) >= CROWDED_GT_COUNT,
    }


candidate_map = {}


def add_candidate(c):

    key = (
        c["image_id"],
        c["x0"],
        c["y0"],
        c["tile_w"],
        c["tile_h"],
    )

    if key not in candidate_map:
        candidate_map[key] = c
    else:
        old_reason = candidate_map[key]["reason"]
        if c["reason"] not in old_reason.split("+"):
            candidate_map[key]["reason"] = old_reason + "+" + c["reason"]


# ------------------------------------------------------------
# A) SMALL-OBJECT CENTERED CANDIDATES
# ------------------------------------------------------------

for image_id, anns in anns_by_image.items():

    im = train_images_by_id[image_id]

    W = int(im["width"])
    H = int(im["height"])
    image_area = max(float(W * H), 1.0)

    for ann in anns:

        x, y, bw, bh = map(float, ann["bbox"])
        rel_area = (bw * bh) / image_area

        if rel_area >= SMALL_AREA_RATIO:
            continue

        cx = x + bw / 2.0
        cy = y + bh / 2.0

        for jx, jy in CENTER_JITTERS:

            x0, y0, tw, th = make_centered_origin(
                cx, cy, W, H, SLICE_SIZE, jx, jy
            )

            c = candidate_from_origin(
                image_id,
                x0,
                y0,
                tw,
                th,
                "small_centered"
            )

            add_candidate(c)


# ------------------------------------------------------------
# B) GRID CANDIDATES FOR CROWDED SOURCES
# ------------------------------------------------------------

crowded_ids = set(
    train_image_df.loc[
        train_image_df["crowded"],
        "image_id"
    ].astype(int)
)

for image_id in sorted(crowded_ids):

    im = train_images_by_id[image_id]

    W = int(im["width"])
    H = int(im["height"])

    tw = min(SLICE_SIZE, W)
    th = min(SLICE_SIZE, H)

    xs = grid_starts(W, tw, GRID_OVERLAP)
    ys = grid_starts(H, th, GRID_OVERLAP)

    for y0 in ys:
        for x0 in xs:

            c = candidate_from_origin(
                image_id,
                x0,
                y0,
                tw,
                th,
                "crowded_grid"
            )

            add_candidate(c)


candidates = list(candidate_map.values())

small_candidates = [
    c for c in candidates
    if c["contains_small"]
]

crowded_positive_candidates = [
    c for c in candidates
    if (
        not c["contains_small"]
        and c["crowded_source"]
        and c["positive"]
        and c["kept_box_count"] >= 2
    )
]

other_positive_candidates = [
    c for c in candidates
    if (
        not c["contains_small"]
        and c["positive"]
        and c not in crowded_positive_candidates
    )
]

negative_candidates = [
    c for c in candidates
    if not c["positive"]
]


def choose(items, max_n):
    if len(items) <= max_n:
        return list(items)
    return rng.sample(items, max_n)


chosen_small = choose(
    small_candidates,
    MAX_SMALL_SLICES
)

chosen_crowded = choose(
    crowded_positive_candidates,
    MAX_CROWDED_SLICES
)

chosen_other = choose(
    other_positive_candidates,
    MAX_OTHER_POSITIVE_SLICES
)

positive_count = (
    len(chosen_small)
    + len(chosen_crowded)
    + len(chosen_other)
)

negative_cap = min(
    MAX_NEGATIVE_SLICES,
    max(1, int(round(positive_count * 0.12)))
)

chosen_negative = choose(
    negative_candidates,
    negative_cap
)

selected_candidates = (
    chosen_small
    + chosen_crowded
    + chosen_other
    + chosen_negative
)

# Deduplicate final selection.
selected_unique = {}
for c in selected_candidates:
    key = (
        c["image_id"],
        c["x0"],
        c["y0"],
        c["tile_w"],
        c["tile_h"],
    )
    selected_unique[key] = c

selected_candidates = list(
    selected_unique.values()
)

print("=" * 105)
print("SMALL-PRIORITY SLICE SELECTION")
print("=" * 105)

print("All unique candidates     :", len(candidates))
print("Small-containing available:", len(small_candidates))
print("Crowded-positive available:", len(crowded_positive_candidates))
print("Other-positive available  :", len(other_positive_candidates))
print("Negative available        :", len(negative_candidates))

print("\nSELECTED")
print("Small-containing slices   :", len(chosen_small))
print("Crowded-positive slices   :", len(chosen_crowded))
print("Other-positive slices     :", len(chosen_other))
print("Negative slices           :", len(chosen_negative))
print("Final unique slices       :", len(selected_candidates))

selected_small_box_occurrences = sum(
    sum(a["original_small"] for a in c["annotations"])
    for c in selected_candidates
)

print(
    "Small-GT occurrences inside selected slices:",
    selected_small_box_occurrences
)

assert len(selected_candidates) > 0
assert len(chosen_small) > 0

print("\nSTATUS: SMALL_PRIORITY_SLICES_SELECTED")


In [ ]:
# ============================================================
# CELL 8 — MATERIALIZE ORIGINAL + SMALL-PRIORITY SLICED DATASET
# ============================================================

if BUILD_ROOT.exists():
    shutil.rmtree(BUILD_ROOT)

for split in ["train", "val"]:
    (BUILD_ROOT / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )
    (BUILD_ROOT / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )


def safe_symlink(src, dst):

    src = Path(src)
    dst = Path(dst)

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    os.symlink(
        str(src),
        str(dst)
    )


def coco_original_to_yolo(
    coco,
    source_img_dir,
    out_img_dir,
    out_label_dir,
    prefix,
):

    local_anns = defaultdict(list)

    for ann in coco["annotations"]:
        local_anns[int(ann["image_id"])].append(ann)

    image_count = 0
    box_count = 0

    for im in coco["images"]:

        image_id = int(im["id"])
        src_name = Path(im["file_name"]).name

        src = Path(source_img_dir) / src_name

        if not src.exists():
            raise FileNotFoundError(src)

        dst_name = f"{prefix}{src_name}"

        safe_symlink(
            src,
            Path(out_img_dir) / dst_name
        )

        W = float(im["width"])
        H = float(im["height"])

        lines = []

        for ann in local_anns[image_id]:

            x, y, bw, bh = map(float, ann["bbox"])

            xc = (x + bw / 2.0) / W
            yc = (y + bh / 2.0) / H
            nw = bw / W
            nh = bh / H

            lines.append(
                f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
            )

            box_count += 1

        label_path = (
            Path(out_label_dir)
            / f"{Path(dst_name).stem}.txt"
        )

        label_path.write_text(
            "\n".join(lines)
            + ("\n" if lines else ""),
            encoding="utf-8"
        )

        image_count += 1

    return image_count, box_count


# ------------------------------------------------------------
# A) ORIGINAL TRAIN — ALL 452
# ------------------------------------------------------------

orig_train_images, orig_train_boxes = coco_original_to_yolo(
    train_coco,
    TRAIN_IMG_DIR,
    BUILD_ROOT / "images" / "train",
    BUILD_ROOT / "labels" / "train",
    "orig__"
)


# ------------------------------------------------------------
# B) ORIGINAL VALIDATION — UNTOUCHED 100 / 354
# ------------------------------------------------------------

val_images, val_boxes = coco_original_to_yolo(
    val_coco,
    VAL_IMG_DIR,
    BUILD_ROOT / "images" / "val",
    BUILD_ROOT / "labels" / "val",
    "val__"
)


# ------------------------------------------------------------
# C) MATERIALIZE SELECTED TRAIN SLICES
# ------------------------------------------------------------

slice_rows = []
slice_total_boxes = 0
slice_small_occurrences = 0

for idx, c in enumerate(selected_candidates):

    src = TRAIN_IMG_DIR / c["file_name"]

    image = cv2.imread(
        str(src),
        cv2.IMREAD_COLOR
    )

    if image is None:
        raise RuntimeError(f"Cannot read {src}")

    x0 = c["x0"]
    y0 = c["y0"]
    tw = c["tile_w"]
    th = c["tile_h"]

    crop = image[
        y0:y0 + th,
        x0:x0 + tw
    ]

    if crop.size == 0:
        raise RuntimeError(
            f"Empty crop: {c}"
        )

    slice_name = (
        f"slice_{idx:04d}"
        f"__img{c['image_id']}"
        f"__x{x0}_y{y0}"
        f".jpg"
    )

    image_path = (
        BUILD_ROOT
        / "images"
        / "train"
        / slice_name
    )

    ok = cv2.imwrite(
        str(image_path),
        crop,
        [int(cv2.IMWRITE_JPEG_QUALITY), 95]
    )

    if not ok:
        raise RuntimeError(
            f"Failed writing {image_path}"
        )

    lines = []

    for ann in c["annotations"]:

        xc = (
            ann["x"] + ann["w"] / 2.0
        ) / tw

        yc = (
            ann["y"] + ann["h"] / 2.0
        ) / th

        nw = ann["w"] / tw
        nh = ann["h"] / th

        lines.append(
            f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}"
        )

        slice_total_boxes += 1

        if ann["original_small"]:
            slice_small_occurrences += 1

    label_path = (
        BUILD_ROOT
        / "labels"
        / "train"
        / f"{Path(slice_name).stem}.txt"
    )

    label_path.write_text(
        "\n".join(lines)
        + ("\n" if lines else ""),
        encoding="utf-8"
    )

    slice_rows.append({
        "slice_name": slice_name,
        "source_image_id": c["image_id"],
        "source_file_name": c["file_name"],
        "x0": x0,
        "y0": y0,
        "tile_w": tw,
        "tile_h": th,
        "reason": c["reason"],
        "positive": c["positive"],
        "box_count": c["kept_box_count"],
        "contains_small": c["contains_small"],
        "small_box_occurrences": int(
            sum(
                a["original_small"]
                for a in c["annotations"]
            )
        ),
        "crowded_source": c["crowded_source"],
    })


slice_manifest_df = pd.DataFrame(
    slice_rows
)

SLICE_MANIFEST = (
    EXP_ROOT
    / "stage4_small_priority_slice_manifest.csv"
)

slice_manifest_df.to_csv(
    SLICE_MANIFEST,
    index=False
)


# ------------------------------------------------------------
# D) DATA YAML
# ------------------------------------------------------------

DATA_YAML = BUILD_ROOT / "data.yaml"

data_yaml = {
    "path": str(BUILD_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "target_coral"
    },
}

with open(
    DATA_YAML,
    "w",
    encoding="utf-8"
) as f:
    yaml.safe_dump(
        data_yaml,
        f,
        sort_keys=False
    )


print("=" * 105)
print("STAGE-4 DATASET CREATED")
print("=" * 105)

print("Original train images :", orig_train_images)
print("Original train boxes  :", orig_train_boxes)
print("Added slices          :", len(slice_manifest_df))
print("Added slice boxes     :", slice_total_boxes)
print("Small box occurrences :", slice_small_occurrences)
print("Final train images    :", orig_train_images + len(slice_manifest_df))
print("Final train boxes     :", orig_train_boxes + slice_total_boxes)
print("Validation images     :", val_images)
print("Validation boxes      :", val_boxes)
print("Data YAML             :", DATA_YAML)

print("\nSTATUS: STAGE4_DATASET_CREATED")


In [ ]:
# ============================================================
# CELL 9 — STRICT DATASET + LEAKAGE VALIDATION
# ============================================================

train_imgs = sorted(
    (BUILD_ROOT / "images" / "train").glob("*")
)

train_labels = sorted(
    (BUILD_ROOT / "labels" / "train").glob("*.txt")
)

val_imgs = sorted(
    (BUILD_ROOT / "images" / "val").glob("*")
)

val_labels = sorted(
    (BUILD_ROOT / "labels" / "val").glob("*.txt")
)

assert len(train_imgs) == len(train_labels)
assert len(val_imgs) == len(val_labels)

assert orig_train_images == 452
assert orig_train_boxes == 1540
assert len(val_imgs) == 100
assert val_boxes == 354

# Validation must contain no generated slices.
assert all(
    p.name.startswith("val__")
    for p in val_imgs
)

assert all(
    not p.name.startswith("slice_")
    for p in val_imgs
)


def count_yolo_boxes(label_dir):

    total = 0
    invalid = []

    for lp in Path(label_dir).glob("*.txt"):

        txt = lp.read_text(
            encoding="utf-8"
        ).strip()

        if not txt:
            continue

        for line in txt.splitlines():

            parts = line.split()

            if len(parts) != 5:
                invalid.append(
                    (lp.name, line)
                )
                continue

            cls, xc, yc, w, h = map(
                float,
                parts
            )

            valid = (
                int(cls) == 0
                and 0 <= xc <= 1
                and 0 <= yc <= 1
                and 0 < w <= 1
                and 0 < h <= 1
            )

            if not valid:
                invalid.append(
                    (lp.name, line)
                )

            total += 1

    return total, invalid


train_box_count, train_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "train"
)

val_box_count, val_invalid = count_yolo_boxes(
    BUILD_ROOT / "labels" / "val"
)

assert not train_invalid, train_invalid[:10]
assert not val_invalid, val_invalid[:10]

assert train_box_count == (
    1540 + slice_total_boxes
)

assert val_box_count == 354

# Source IDs used for slices must be train IDs only.
train_ids = {
    int(im["id"])
    for im in train_coco["images"]
}

val_ids = {
    int(im["id"])
    for im in val_coco["images"]
}

slice_source_ids = set(
    slice_manifest_df[
        "source_image_id"
    ].astype(int)
)

assert slice_source_ids.issubset(train_ids)

# Image IDs can numerically overlap across independent COCO files,
# so file-name split isolation is the stronger practical check here.
train_original_names = {
    Path(im["file_name"]).name
    for im in train_coco["images"]
}

val_original_names = {
    Path(im["file_name"]).name
    for im in val_coco["images"]
}

assert train_original_names.isdisjoint(
    val_original_names
)

BUILD_SUMMARY = {
    "experiment": "YOLO11n_P2_Stage4_SmallPriority",
    "original_train_images": 452,
    "original_train_boxes": 1540,
    "small_definition": "original bbox area < 1% image area",
    "crowded_definition": ">=4 GT objects per image",
    "slice_size": SLICE_SIZE,
    "grid_overlap": GRID_OVERLAP,
    "minimum_visible_fraction": MIN_VISIBLE_FRACTION,
    "small_priority_slices": int(
        slice_manifest_df["contains_small"].sum()
    ),
    "all_added_slices": int(
        len(slice_manifest_df)
    ),
    "added_slice_boxes": int(
        slice_total_boxes
    ),
    "small_box_occurrences_in_slices": int(
        slice_small_occurrences
    ),
    "final_train_images": int(
        len(train_imgs)
    ),
    "final_train_boxes": int(
        train_box_count
    ),
    "validation_images": int(
        len(val_imgs)
    ),
    "validation_boxes": int(
        val_box_count
    ),
    "test_used": False,
}

BUILD_SUMMARY_JSON = (
    EXP_ROOT
    / "stage4_dataset_build_summary.json"
)

with open(
    BUILD_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        BUILD_SUMMARY,
        f,
        indent=2
    )

print("=" * 105)
print("STRICT STAGE-4 DATASET VALIDATION PASSED")
print("=" * 105)

for k, v in BUILD_SUMMARY.items():
    print(f"{k:<40}: {v}")

print("\nNO VALIDATION SLICES.")
print("NO TEST DATA USED.")
print("STATUS: STAGE4_DATASET_VALIDATED")


In [ ]:
# ============================================================
# CELL 10 — STAGE-4 TRAINING CONFIG
# P2 + SMALL-PRIORITY SLICES + DETAIL-PRESERVING AUGMENTATION
# ============================================================

# P2 increases high-resolution feature computation.
# T4-class GPUs use a conservative batch by default.
BATCH_STAGE4 = 6 if gpu_gb < 20 else 8

EPOCHS_STAGE4 = 15
IMGSZ_STAGE4 = 960

LR0_STAGE4 = 1.0e-4
LRF_STAGE4 = 0.10

stage4_augmentations = [
    A.RandomBrightnessContrast(
        brightness_limit=(-0.08, 0.08),
        contrast_limit=(-0.10, 0.10),
        p=0.15,
    ),
    A.CLAHE(
        clip_limit=(1.0, 1.6),
        tile_grid_size=(8, 8),
        p=0.03,
    ),
]

STAGE4_CONFIG = {
    "architecture": "YOLO11n-P2",
    "outputs": ["P2/4", "P3/8", "P4/16", "P5/32"],
    "starting_checkpoint": str(STAGE3_BEST),
    "epochs": EPOCHS_STAGE4,
    "imgsz": IMGSZ_STAGE4,
    "batch": BATCH_STAGE4,
    "optimizer": "AdamW",
    "lr0": LR0_STAGE4,
    "lrf": LRF_STAGE4,
    "weight_decay": 0.0005,
    "cos_lr": True,
    "warmup_epochs": 1.0,
    "multi_scale": 0.0,
    "mosaic": 0.05,
    "mixup": 0.0,
    "cutmix": 0.0,
    "degrees": 4.0,
    "translate": 0.04,
    "scale": 0.08,
    "fliplr": 0.50,
    "flipud": 0.05,
    "blur": False,
    "noise": False,
    "downscale": False,
    "compression": False,
    "test_used": False,
}

CONFIG_JSON = (
    EXP_ROOT
    / "stage4_experiment_config.json"
)

with open(
    CONFIG_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        STAGE4_CONFIG,
        f,
        indent=2
    )

print("=" * 105)
print("STAGE-4 TRAINING CONFIG")
print("=" * 105)

for k, v in STAGE4_CONFIG.items():
    print(f"{k:<28}: {v}")

print("\nSmall-detail degradation:")
print("Blur        : OFF")
print("Noise       : OFF")
print("Downscale   : OFF")
print("Compression : OFF")

print("\nSTATUS: STAGE4_CONFIG_READY")


In [ ]:
# ============================================================
# CELL 11 — TRAIN YOLO11n-P2 STAGE 4
# ============================================================

RUN_NAME = "Detection_Stage4_P2"

print("=" * 115)
print("YOLO11n-P2 — STAGE 4 SMALL-PRIORITY FINE-TUNING")
print("=" * 115)

print("Architecture       : P2/P3/P4/P5")
print("Strides            : 4/8/16/32")
print("Start checkpoint   :", STAGE3_BEST)
print("Train data         : Original 452 + Small-Priority slices")
print("Validation         : Original 100 only")
print("Epochs             :", EPOCHS_STAGE4)
print("Image size         :", IMGSZ_STAGE4)
print("Batch              :", BATCH_STAGE4)
print("LR                 :", LR0_STAGE4)
print("Multi-scale        : OFF")
print("Mosaic             : 0.05")
print("Blur/Noise/Downscale: OFF")
print("Test               : NOT USED")

start_time = time.time()

train_result = p2_model.train(

    data=str(DATA_YAML),

    epochs=EPOCHS_STAGE4,
    imgsz=IMGSZ_STAGE4,
    batch=BATCH_STAGE4,

    device=0,
    workers=2,

    optimizer="AdamW",
    lr0=LR0_STAGE4,
    lrf=LRF_STAGE4,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=1.0,

    multi_scale=0.0,

    degrees=4.0,
    translate=0.04,
    scale=0.08,
    shear=0.0,
    perspective=0.0,

    fliplr=0.50,
    flipud=0.05,

    mosaic=0.05,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    close_mosaic=5,

    hsv_h=0.003,
    hsv_s=0.08,
    hsv_v=0.07,

    augmentations=stage4_augmentations,

    box=7.5,
    cls=0.5,
    dfl=1.5,

    amp=True,

    seed=SEED,
    deterministic=True,

    patience=100,
    cache=False,

    save=True,
    save_period=3,
    plots=True,

    project=str(EXP_ROOT),
    name=RUN_NAME,
    exist_ok=True,

    verbose=True,
)

elapsed_min = (
    time.time() - start_time
) / 60.0

RUN_DIR = Path(
    train_result.save_dir
)

BEST_STAGE4 = (
    RUN_DIR
    / "weights"
    / "best.pt"
)

LAST_STAGE4 = (
    RUN_DIR
    / "weights"
    / "last.pt"
)

assert BEST_STAGE4.exists()

print("\n" + "=" * 105)
print("STAGE-4 TRAINING COMPLETE")
print("=" * 105)

print(f"Training time : {elapsed_min:.2f} min")
print("Run directory :", RUN_DIR)
print("best.pt       :", BEST_STAGE4)
print("last.pt       :", LAST_STAGE4)

print("\nTEST WAS NOT EVALUATED.")
print("STATUS: STAGE4_TRAINING_COMPLETE")


In [ ]:
# ============================================================
# CELL 12 — STANDARD ULTRALYTICS VALIDATION
# ORIGINAL 100 VALIDATION IMAGES ONLY
# ============================================================

stage4_model = YOLO(
    str(BEST_STAGE4)
)

metrics_stage4 = stage4_model.val(

    data=str(DATA_YAML),

    split="val",

    imgsz=960,
    batch=BATCH_STAGE4,

    device=0,

    conf=0.001,
    iou=0.70,
    max_det=300,

    plots=True,
    verbose=True,
)

ULTRA_STAGE4 = {
    "precision": float(metrics_stage4.box.mp),
    "recall": float(metrics_stage4.box.mr),
    "map50": float(metrics_stage4.box.map50),
    "map50_95": float(metrics_stage4.box.map),
}

print("=" * 100)
print("STAGE-4 ULTRALYTICS VALIDATION")
print("=" * 100)

for k, v in ULTRA_STAGE4.items():
    print(f"{k:<15}: {v:.6f}")

print("\nTEST WAS NOT EVALUATED.")
print("STATUS: STAGE4_ULTRALYTICS_VALIDATION_COMPLETE")


In [ ]:
# ============================================================
# CELL 13 — COMMON OBJECT-LEVEL AUDIT FUNCTIONS
# ============================================================

with open(
    VAL_COCO,
    "r",
    encoding="utf-8"
) as f:
    audit_val_coco = json.load(f)

val_image_id_by_basename = {
    Path(im["file_name"]).name: int(im["id"])
    for im in audit_val_coco["images"]
}

val_image_meta = {
    int(im["id"]): im
    for im in audit_val_coco["images"]
}

val_gt_by_id = defaultdict(list)

for ann in audit_val_coco["annotations"]:

    image_id = int(
        ann["image_id"]
    )

    x, y, w, h = map(
        float,
        ann["bbox"]
    )

    val_gt_by_id[image_id].append({
        "bbox": np.array(
            [x, y, x + w, y + h],
            dtype=np.float32
        ),
        "area": float(w * h),
    })


def iou_matrix(a, b):

    if len(a) == 0 or len(b) == 0:
        return np.zeros(
            (len(a), len(b)),
            dtype=np.float32
        )

    a = np.asarray(
        a,
        dtype=np.float32
    )

    b = np.asarray(
        b,
        dtype=np.float32
    )

    lt = np.maximum(
        a[:, None, :2],
        b[None, :, :2]
    )

    rb = np.minimum(
        a[:, None, 2:],
        b[None, :, 2:]
    )

    wh = np.clip(
        rb - lt,
        0,
        None
    )

    inter = (
        wh[..., 0]
        * wh[..., 1]
    )

    aa = (
        np.clip(a[:, 2] - a[:, 0], 0, None)
        * np.clip(a[:, 3] - a[:, 1], 0, None)
    )

    bb = (
        np.clip(b[:, 2] - b[:, 0], 0, None)
        * np.clip(b[:, 3] - b[:, 1], 0, None)
    )

    union = (
        aa[:, None]
        + bb[None, :]
        - inter
    )

    return inter / np.clip(
        union,
        1e-9,
        None
    )


def greedy_match(
    pred_boxes,
    gt_boxes,
    threshold=0.50
):

    if len(pred_boxes) == 0:
        return [], [], list(
            range(len(gt_boxes))
        )

    if len(gt_boxes) == 0:
        return [], list(
            range(len(pred_boxes))
        ), []

    ious = iou_matrix(
        pred_boxes,
        gt_boxes
    )

    candidates = []

    for p in range(len(pred_boxes)):
        for g in range(len(gt_boxes)):

            if ious[p, g] >= threshold:

                candidates.append(
                    (
                        float(ious[p, g]),
                        p,
                        g
                    )
                )

    candidates.sort(
        reverse=True
    )

    used_p = set()
    used_g = set()
    matches = []

    for iou, p, g in candidates:

        if p in used_p or g in used_g:
            continue

        used_p.add(p)
        used_g.add(g)

        matches.append(
            (p, g, iou)
        )

    unmatched_p = [
        p for p in range(len(pred_boxes))
        if p not in used_p
    ]

    unmatched_g = [
        g for g in range(len(gt_boxes))
        if g not in used_g
    ]

    return (
        matches,
        unmatched_p,
        unmatched_g
    )


def size_bucket(
    gt_box,
    image_w,
    image_h
):

    x1, y1, x2, y2 = map(
        float,
        gt_box
    )

    ratio = (
        max(0.0, x2 - x1)
        * max(0.0, y2 - y1)
        / max(
            float(image_w * image_h),
            1.0
        )
    )

    if ratio < 0.01:
        return "small"

    if ratio < 0.10:
        return "medium"

    return "large"


def threshold_sweep(
    pred_dict
):

    rows = []

    for conf in np.round(
        np.arange(
            0.05,
            0.81,
            0.025
        ),
        3
    ):

        TP = FP = FN = 0

        for image_name, pred in pred_dict.items():

            image_id = (
                val_image_id_by_basename[
                    image_name
                ]
            )

            gt_boxes = np.array(
                [
                    x["bbox"]
                    for x in val_gt_by_id[
                        image_id
                    ]
                ],
                dtype=np.float32
            ).reshape(-1, 4)

            keep = (
                pred["conf"]
                >= conf
            )

            pred_boxes = (
                pred["boxes"][keep]
            )

            matches, up, ug = greedy_match(
                pred_boxes,
                gt_boxes,
                0.50
            )

            TP += len(matches)
            FP += len(up)
            FN += len(ug)

        precision = (
            TP / (TP + FP)
            if (TP + FP)
            else 0.0
        )

        recall = (
            TP / (TP + FN)
            if (TP + FN)
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        rows.append({
            "confidence": float(conf),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    df = pd.DataFrame(rows)

    best = df.loc[
        df["f1"].idxmax()
    ]

    return df, best


def detailed_metrics(
    pred_dict,
    conf
):

    gt_rows = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    crowded_totals = {
        False: {
            "TP": 0,
            "FP": 0,
            "FN": 0,
            "GT": 0,
            "images": 0,
        },
        True: {
            "TP": 0,
            "FP": 0,
            "FN": 0,
            "GT": 0,
            "images": 0,
        },
    }

    for image_name, pred in pred_dict.items():

        image_id = (
            val_image_id_by_basename[
                image_name
            ]
        )

        meta = val_image_meta[
            image_id
        ]

        gt_items = val_gt_by_id[
            image_id
        ]

        gt_boxes = np.array(
            [
                x["bbox"]
                for x in gt_items
            ],
            dtype=np.float32
        ).reshape(-1, 4)

        keep = (
            pred["conf"]
            >= conf
        )

        pred_boxes = (
            pred["boxes"][keep]
        )

        matches, up, ug = greedy_match(
            pred_boxes,
            gt_boxes,
            0.50
        )

        matched_gt = {
            g: (p, iou)
            for p, g, iou in matches
        }

        for g, gt in enumerate(
            gt_items
        ):

            bucket = size_bucket(
                gt["bbox"],
                int(meta["width"]),
                int(meta["height"])
            )

            gt_rows.append({
                "image": image_name,
                "size": bucket,
                "detected": g in matched_gt,
                "iou": (
                    float(
                        matched_gt[g][1]
                    )
                    if g in matched_gt
                    else 0.0
                ),
            })

        tp = len(matches)
        fp = len(up)
        fn = len(ug)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        crowded = (
            len(gt_items)
            >= CROWDED_GT_COUNT
        )

        crowded_totals[
            crowded
        ]["TP"] += tp

        crowded_totals[
            crowded
        ]["FP"] += fp

        crowded_totals[
            crowded
        ]["FN"] += fn

        crowded_totals[
            crowded
        ]["GT"] += len(gt_items)

        crowded_totals[
            crowded
        ]["images"] += 1

    gt_df = pd.DataFrame(
        gt_rows
    )

    size_rows = []

    for bucket in [
        "small",
        "medium",
        "large"
    ]:

        g = gt_df[
            gt_df["size"]
            == bucket
        ]

        size_rows.append({
            "size": bucket,
            "GT": len(g),
            "TP": int(
                g["detected"].sum()
            ),
            "FN": int(
                (~g["detected"]).sum()
            ),
            "recall": (
                float(
                    g["detected"].mean()
                )
                if len(g)
                else np.nan
            ),
            "mean_matched_iou": (
                float(
                    g.loc[
                        g["detected"],
                        "iou"
                    ].mean()
                )
                if g["detected"].any()
                else np.nan
            ),
        })

    crowded_rows = []

    for crowded in [
        False,
        True
    ]:

        d = crowded_totals[
            crowded
        ]

        precision = (
            d["TP"]
            / (d["TP"] + d["FP"])
            if (
                d["TP"] + d["FP"]
            )
            else 0.0
        )

        recall = (
            d["TP"]
            / (d["TP"] + d["FN"])
            if (
                d["TP"] + d["FN"]
            )
            else 0.0
        )

        crowded_rows.append({
            "crowded": crowded,
            "images": d["images"],
            "GT": d["GT"],
            "TP": d["TP"],
            "FP": d["FP"],
            "FN": d["FN"],
            "precision": precision,
            "recall": recall,
        })

    p_all = (
        total_tp
        / (total_tp + total_fp)
        if (
            total_tp + total_fp
        )
        else 0.0
    )

    r_all = (
        total_tp
        / (total_tp + total_fn)
        if (
            total_tp + total_fn
        )
        else 0.0
    )

    f1_all = (
        2 * p_all * r_all
        / (p_all + r_all)
        if (p_all + r_all)
        else 0.0
    )

    overall = {
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "precision": p_all,
        "recall": r_all,
        "f1": f1_all,
    }

    return (
        overall,
        pd.DataFrame(size_rows),
        pd.DataFrame(crowded_rows),
        gt_df,
    )


print("STATUS: AUDIT_FUNCTIONS_READY")


In [ ]:
# ============================================================
# CELL 14 — STAGE-4 STANDARD INFERENCE + THRESHOLD SWEEP
# ============================================================

original_val_paths = sorted([
    p
    for p in VAL_IMG_DIR.iterdir()
    if p.suffix.lower()
    in {
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".bmp",
    }
])

assert len(original_val_paths) == 100

t0 = time.time()

stage4_results = stage4_model.predict(

    source=[
        str(p)
        for p in original_val_paths
    ],

    imgsz=960,

    conf=0.05,
    iou=0.70,
    max_det=300,

    device=0,

    verbose=False,
)

wall_seconds = (
    time.time() - t0
)

stage4_predictions = {}

model_inference_ms = []

for image_path, result in zip(
    original_val_paths,
    stage4_results
):

    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):

        boxes = np.empty(
            (0, 4),
            dtype=np.float32
        )

        conf = np.empty(
            (0,),
            dtype=np.float32
        )

    else:

        boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        conf = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

    stage4_predictions[
        image_path.name
    ] = {
        "boxes": boxes,
        "conf": conf,
    }

    if (
        hasattr(result, "speed")
        and result.speed
        and "inference" in result.speed
    ):
        model_inference_ms.append(
            float(
                result.speed["inference"]
            )
        )


stage4_sweep, stage4_best = threshold_sweep(
    stage4_predictions
)

STAGE4_BEST_CONF = float(
    stage4_best["confidence"]
)

(
    stage4_overall,
    stage4_size,
    stage4_crowded,
    stage4_gt_df,
) = detailed_metrics(
    stage4_predictions,
    STAGE4_BEST_CONF,
)

mean_inference_ms = (
    float(np.mean(model_inference_ms))
    if model_inference_ms
    else None
)

wall_ms_per_image = (
    wall_seconds
    / len(original_val_paths)
    * 1000.0
)

print("=" * 105)
print("STAGE-4 P2 CUSTOM VALIDATION AUDIT")
print("=" * 105)

print("Best confidence :", STAGE4_BEST_CONF)
print("Overall         :", stage4_overall)

print("\nBy size:")
print(
    stage4_size.to_string(
        index=False
    )
)

print("\nCrowded:")
print(
    stage4_crowded.to_string(
        index=False
    )
)

print("\nTiming:")
print(
    "Mean model inference ms/image:",
    mean_inference_ms
)
print(
    "Wall-clock ms/image:",
    wall_ms_per_image
)

print("\nSTATUS: STAGE4_CUSTOM_AUDIT_COMPLETE")


In [ ]:
# ============================================================
# CELL 15 — FINAL STAGE2 vs STAGE3 vs STAGE4 COMPARISON
#
# Stage2 FP/FN here use the SAME standard IoU>=0.5 matching
# convention as Stage3/Stage4:
# LOW_IOU pairs count as one FP + one FN.
# ============================================================

stage2_ref = {
    "method": "Stage2 Standard",
    "precision": 0.7884615385,
    "overall_recall": 0.6949152542,
    "small_recall": 0.3611111111,
    "medium_recall": 0.6745283019,
    "large_recall": 0.8490566038,
    "crowded_recall": 0.6567796610,
    "TP": 246,
    "FP": 66,
    "FN": 108,
    "best_conf": 0.325,
    "map50": 0.7727,
    "map50_95": 0.4439,
}

stage3_ref = {
    "method": "Stage3 Standard",
    "precision": 0.7937,
    "overall_recall": 0.7175,
    "small_recall": 0.4167,
    "medium_recall": 0.6981,
    "large_recall": 0.8585,
    "crowded_recall": 0.6822,
    "TP": 254,
    "FP": 66,
    "FN": 100,
    "best_conf": 0.350,
    "map50": 0.793,
    "map50_95": 0.473,
}


def value_for_size(
    size_df,
    bucket
):

    return float(
        size_df.loc[
            size_df["size"] == bucket,
            "recall"
        ].iloc[0]
    )


stage4_ref = {
    "method": "Stage4 YOLO11n-P2",
    "precision": stage4_overall["precision"],
    "overall_recall": stage4_overall["recall"],
    "small_recall": value_for_size(
        stage4_size,
        "small"
    ),
    "medium_recall": value_for_size(
        stage4_size,
        "medium"
    ),
    "large_recall": value_for_size(
        stage4_size,
        "large"
    ),
    "crowded_recall": float(
        stage4_crowded.loc[
            stage4_crowded["crowded"] == True,
            "recall"
        ].iloc[0]
    ),
    "TP": stage4_overall["TP"],
    "FP": stage4_overall["FP"],
    "FN": stage4_overall["FN"],
    "best_conf": STAGE4_BEST_CONF,
    "map50": ULTRA_STAGE4["map50"],
    "map50_95": ULTRA_STAGE4["map50_95"],
}


comparison_df = pd.DataFrame([
    stage2_ref,
    stage3_ref,
    stage4_ref,
])

COMPARISON_CSV = (
    EXP_ROOT
    / "stage2_stage3_stage4_p2_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_CSV,
    index=False
)

SWEEP_CSV = (
    EXP_ROOT
    / "stage4_confidence_threshold_sweep.csv"
)

stage4_sweep.to_csv(
    SWEEP_CSV,
    index=False
)

SIZE_CSV = (
    EXP_ROOT
    / "stage4_performance_by_object_size.csv"
)

stage4_size.to_csv(
    SIZE_CSV,
    index=False
)

CROWD_CSV = (
    EXP_ROOT
    / "stage4_performance_crowded_vs_noncrowded.csv"
)

stage4_crowded.to_csv(
    CROWD_CSV,
    index=False
)


print("=" * 145)
print("FINAL DETECTOR COMPARISON")
print("=" * 145)

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "precision": "{:.4f}".format,
            "overall_recall": "{:.4f}".format,
            "small_recall": "{:.4f}".format,
            "medium_recall": "{:.4f}".format,
            "large_recall": "{:.4f}".format,
            "crowded_recall": "{:.4f}".format,
            "best_conf": "{:.3f}".format,
            "map50": "{:.4f}".format,
            "map50_95": "{:.4f}".format,
        }
    )
)

print("\nStage3 → Stage4 deltas")

for metric in [
    "precision",
    "overall_recall",
    "small_recall",
    "medium_recall",
    "large_recall",
    "crowded_recall",
    "map50",
    "map50_95",
]:

    old = float(
        comparison_df.loc[
            comparison_df["method"]
            == "Stage3 Standard",
            metric
        ].iloc[0]
    )

    new = float(
        comparison_df.loc[
            comparison_df["method"]
            == "Stage4 YOLO11n-P2",
            metric
        ].iloc[0]
    )

    print(
        f"{metric:<18}: {new - old:+.4f}"
    )

print("\nSaved:", COMPARISON_CSV)
print("TEST SET WAS NOT USED.")
print("STATUS: STAGE4_COMPARISON_COMPLETE")


In [ ]:
# ============================================================
# CELL 16 — FINAL DECISION SUMMARY + SAVE JSON
# ============================================================

stage4_small_recall = value_for_size(
    stage4_size,
    "small"
)

stage4_crowded_recall = float(
    stage4_crowded.loc[
        stage4_crowded["crowded"] == True,
        "recall"
    ].iloc[0]
)

success_checks = {
    "small_recall_ge_0_50":
        bool(stage4_small_recall >= 0.50),

    "crowded_recall_ge_0_72":
        bool(stage4_crowded_recall >= 0.72),

    "overall_recall_ge_0_72":
        bool(stage4_overall["recall"] >= 0.72),

    "precision_ge_0_75":
        bool(stage4_overall["precision"] >= 0.75),

    "map50_95_ge_stage3":
        bool(
            ULTRA_STAGE4["map50_95"]
            >= 0.473
        ),

    "large_recall_not_majorly_degraded":
        bool(
            value_for_size(
                stage4_size,
                "large"
            )
            >= 0.82
        ),
}

summary = {
    "experiment":
        "YOLO11n_P2_Stage4_SmallPriority",

    "model":
        str(BEST_STAGE4),

    "architecture_yaml":
        str(P2_YAML),

    "strides":
        strides,

    "parameters":
        int(P2_PARAMS),

    "best_operating_confidence":
        float(STAGE4_BEST_CONF),

    "precision":
        float(stage4_overall["precision"]),

    "overall_recall":
        float(stage4_overall["recall"]),

    "f1":
        float(stage4_overall["f1"]),

    "small_recall":
        float(stage4_small_recall),

    "medium_recall":
        float(
            value_for_size(
                stage4_size,
                "medium"
            )
        ),

    "large_recall":
        float(
            value_for_size(
                stage4_size,
                "large"
            )
        ),

    "crowded_recall":
        float(stage4_crowded_recall),

    "TP":
        int(stage4_overall["TP"]),

    "FP":
        int(stage4_overall["FP"]),

    "FN":
        int(stage4_overall["FN"]),

    "map50":
        float(ULTRA_STAGE4["map50"]),

    "map50_95":
        float(ULTRA_STAGE4["map50_95"]),

    "mean_model_inference_ms_per_image":
        mean_inference_ms,

    "wall_ms_per_image":
        float(wall_ms_per_image),

    "success_checks":
        success_checks,

    "test_evaluated":
        False,
}

SUMMARY_JSON = (
    EXP_ROOT
    / "stage4_final_summary.json"
)

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )


print("=" * 110)
print("STAGE-4 FINAL SUMMARY")
print("=" * 110)

for k, v in summary.items():

    if k == "success_checks":
        continue

    print(
        f"{k:<38}: {v}"
    )

print("\nSUCCESS CHECKS")

for k, v in success_checks.items():
    print(
        f"{k:<40}: {'PASS' if v else 'NOT MET'}"
    )

print("\nInterpretation rules:")
print(
    "- Primary target: Small Recall improvement."
)
print(
    "- Do not accept a small-recall gain if precision or large-object recall collapses."
)
print(
    "- mAP50:95 remains the main overall localization/detection quality metric."
)
print(
    "- P2 latency must be compared with Stage3 because real-time deployment remains a project requirement."
)

print("\nSaved:", SUMMARY_JSON)
print("Internal TEST remains untouched.")
print("\nSTATUS: YOLO11N_P2_STAGE4_COMPLETE")


## How to decide after Stage 4

Primary success target:

- **Small Recall >= 0.50**

Secondary targets:

- Crowded Recall >= 0.72
- Overall Recall >= 0.72
- Precision >= 0.75
- mAP50:95 >= Stage 3 (~0.473)
- Large Recall should remain healthy
- P2 latency must still be compatible with the intended deployment mode

### If P2 wins

Freeze this detector candidate and compare it against RF-DETR using the same validation audit. Do not open the internal test until detector model selection is complete.

### If P2 raises Small Recall but hurts speed

Keep two candidates:

- **Real-time candidate:** Stage 3 YOLO11n
- **Small-object accuracy candidate:** Stage 4 YOLO11n-P2

### If P2 does not materially improve Small Recall

The next priority is not more random epochs. Audit the actual pixel width/height distribution of the missed small GT, inspect annotation tightness, and consider targeted real-data collection or a controlled small-object loss/assignment experiment.
